# Chapter 5: The Expected Method

Code of Chapter 5 of *Practical Introduction to Non-Life Insurance Actuarial Mathematics* (F.-X. Cart-Tanneur). Run the cells from the top: later cells reuse the data and imports of earlier ones. Each comment after `#` gives the value printed in the book.

In [ ]:
# Setup: make the book's modules importable (works locally and in the browser)
import os, sys
sys.path.insert(0, os.path.abspath(os.path.join("..", "src")))

## 5.1.2 Worked example

In [ ]:
import numpy as np
import pandas as pd
from giexpected import expected_ultimate, actual_vs_expected
from gidevelop import average_factors, cumulative_factors, develop, latest_diagonal

T = pd.DataFrame([[1000, 1500, 1650, 1700], [1100, 1700, 1850, np.nan],
                  [1200, 1750, np.nan, np.nan], [1300, np.nan, np.nan, np.nan]],
                 index=pd.Index([2022, 2023, 2024, 2025], name="accident_year"),
                 columns=pd.Index([12, 24, 36, 48], name="age"))
EP = pd.Series([2500, 2700, 2900, 3100.0], index=T.index)
eu = expected_ultimate(EP, 0.70)                   # 1750, 1890, 2030, 2170
(eu - latest_diagonal(T)["latest"]).sum()          # 1240.0

## 5.3.1 Setting the expected pure premium

In [ ]:
from gitriangle import synthetic_history, triangle, exposure_by_accident_year
from giexpected import trended_ratios

pol, cl, txs = synthetic_history(2020, 2025, seed=100)
I = triangle(cl, txs, "incurred", 2021, 2025)
P = triangle(cl, txs, "paid", 2021, 2025)
E = exposure_by_accident_year(pol, range(2021, 2026))
dI = develop(I, cumulative_factors(average_factors(I, "volume"), 1.0))
mature = dI["ultimate"].loc[2021:2023]
pp25 = trended_ratios(mature, E["earned_exposure"].loc[2021:2023], 0.04, 2025)
pp25.round(2).tolist(), pp25.mean().round(2)       # [531.26, 450.62, 449.58], 477.15

## 5.3.2 Results, and a comparison with the truth

In [ ]:
pp = {y: trended_ratios(mature, E["earned_exposure"].loc[2021:2023], 0.04, y).mean()
      for y in (2024, 2025)}                       # 458.80, 477.15
eu_s = pd.Series({y: E.loc[y, "earned_exposure"] * pp[y] for y in (2024, 2025)})
eu_s.round(0).tolist()                             # [2744727, 2964160]

## 5.4.2 The example triangle

In [ ]:
ave = actual_vs_expected(eu, latest_diagonal(T)["latest"], develop(T, cumulative_factors(
    average_factors(T, "volume"), 1.0))["cdf"])
ave["ratio"].round(3).tolist()                     # [0.971, 1.008, 0.971, 1.013]

## 5.4.3 The synthetic portfolio

In [ ]:
from giexpected import actual_vs_expected
li = latest_diagonal(I)["latest"].loc[[2024, 2025]]
actual_vs_expected(eu_s, li, dI["cdf"].loc[[2024, 2025]])["ratio"].round(3).tolist()   # [1.07, 1.083]

## Exercise 1: A lower expected loss ratio

In [ ]:
(expected_ultimate(EP, 0.65) - latest_diagonal(T)["latest"]).tolist()   # [-75, -95, 135, 715]

## Exercise 3: Reading the synthetic A/E ratios

In [ ]:
(I.loc[2025, 12] * dI.loc[2025, "cdf"]) / E.loc[2025, "earned_exposure"]   # 516.61

## 5.7 Python Walkthrough

In [ ]:
from giexpected import expected_ultimate, trended_ratios, expected_reserves, actual_vs_expected

# Section 5.1: expected method on the example triangle
expected_ultimate(EP, 0.70).sum()                  # 7840.0

# Section 5.3: synthetic portfolio, exposure base with 4% trend
pp24 = trended_ratios(mature, E["earned_exposure"].loc[2021:2023], 0.04, 2024).mean()   # 458.80
pp25 = trended_ratios(mature, E["earned_exposure"].loc[2021:2023], 0.04, 2025).mean()   # 477.15
lp = latest_diagonal(P)["latest"].loc[[2024, 2025]]
expected_reserves(eu_s, lp, li)["ibnr"].round(0).tolist()   # [-163809, 125419]

# Section 5.4: actual versus expected on paid claims
dP = develop(P, cumulative_factors(average_factors(P, "volume"), 1.0))
actual_vs_expected(eu_s, lp, dP["cdf"].loc[[2024, 2025]])["ratio"].round(3).tolist()   # [1.048, 1.037]